### Highway 15 Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import scipy.stats as ss
from sklearn.preprocessing import MinMaxScaler
import scipy.fft
import pywt
from scipy.signal.windows import hann
from scipy import interpolate
from scipy.ndimage import uniform_filter
from scipy.signal import ShortTimeFFT
from scipy.signal import savgol_filter
from scipy.signal import hilbert
import ssqueezepy as sq
from ssqueezepy.experimental import scale_to_freq

#### DataFrame --> File Name Legend:
- df --> hwy15_2025_001.asc
- df2 --> hwy15_2025_2_001.asc
- df3 --> hwy15_2025_3_001.asc

In [ ]:
column_names = ['NS', 'EW', 'Z', 'nsL', 'ewL', 'zL', 'aY', 'aX', 'aZ']
df = pd.read_table('hwy15_2025_001.asc', delimiter=r'\s+', encoding='latin-1', skiprows=32, names=column_names)
#df.head().style

In [ ]:
column_names2 = ['NS', 'EW', 'Z', 'nsL', 'ewL', 'zL', 'aY', 'aX', 'aZ']
df2 = pd.read_table('hwy15_2025_2_001.asc', delimiter=r'\s+', encoding='latin-1', skiprows=32, names=column_names2)
#df2.head().style

In [ ]:
column_names3 = ['NS', 'EW', 'Z', 'nsL', 'ewL', 'zL', 'aY', 'aX', 'aZ']
df3 = pd.read_table('hwy15_2025_3_001.asc', delimiter=r'\s+', encoding='latin-1', skiprows=32, names=column_names3)
#df3.head().style

In [ ]:
any_null = df.isnull().any().any()
print(any_null) # Will show false if no null values
#df.info()
print(df2.isnull().any().any())
print(df3.isnull().any().any())

In [ ]:
print(df.columns)
# Want to make a time column:
sampling_rate = 1024  # Hz (same for all data files)
dataframes = [df, df2, df3]
for i, dataframe in enumerate(dataframes):
    time = dataframe.index/sampling_rate
    dataframe['time (s)'] = time
    print(dataframe.head())

***
### Plotting the Time Series
##### Notes:
- Plot the NS, EW, and Z data separately

In [ ]:
for i, dframe in enumerate(dataframes):
    if i==0:
        print("dataframe: df")
    else:
        print(f"dataframe: df{i+1}")
    # Plot:
    fig, axes = plt.subplots(3, 1, figsize=(10, 10))
    time_minutes = dframe['time (s)']/60
    # North-South data:
    axes[0].plot(time_minutes, dframe['NS'], label='North-South', color='forestgreen')
    axes[0].set_ylabel('Amplitude')
    axes[0].set_title('North-South Data')
    axes[0].set_xlabel('Time (min)')
    axes[0].grid(True)
    
    # East-West data:
    axes[1].plot(time_minutes, dframe['EW'], label='East-West', color='rebeccapurple')
    axes[1].set_ylabel('Amplitude')
    axes[1].set_title('East-West Data')
    axes[1].set_xlabel('Time (min)')
    axes[1].grid(True)
    
    # Z data:
    axes[2].plot(time_minutes, dframe['Z'], label='Z/Vertical Component', color='royalblue')
    axes[2].set_ylabel('Amplitude')
    axes[2].set_title('Z-Component Data')
    axes[2].set_xlabel('Time (min)')
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
original_sampling_rate = 1024

***
### Normalizing, Detrending and Shifting Data
##### Notes:
- data range of time series was normalized using MinMaxScaler from sklearn
- data was linearly detrended using signal.detrend from scipy
- the mean of the data was shifted to approximately zero

In [ ]:
for i, dframe in enumerate(dataframes):
    if i==0:
        print("dataframe: df")
    else:
        print(f"dataframe: df{i+1}")

    # Linearly detrend data, normalize and shift to zero mean:
    # Detrend
    dframe['NS_Detrended'] = signal.detrend(dframe['NS'], type='linear') # North-South data
    dframe['EW_Detrended'] = signal.detrend(dframe['EW'], type='linear') # East-West data
    dframe['Z_Detrended'] = signal.detrend(dframe['Z'], type='linear') # Z-component data
    
    # Normalize
    scaler = MinMaxScaler()
    dframe['NS_Normalized'] = scaler.fit_transform(dframe['NS_Detrended'].values.reshape(-1, 1)).flatten() # North-South
    dframe['EW_Normalized'] = scaler.fit_transform(dframe['EW_Detrended'].values.reshape(-1, 1)).flatten() # East-West
    dframe['Z_Normalized'] = scaler.fit_transform(dframe['Z_Detrended'].values.reshape(-1, 1)).flatten() # Z-component
    print(f"NS Min-Max normalized range: [{dframe['NS_Normalized'].min():.3f}, {dframe['NS_Normalized'].max():.3f}]")
    print(f"EW Min-Max normalized range: [{dframe['EW_Normalized'].min():.3f}, {dframe['EW_Normalized'].max():.3f}]")
    print(f"Z Min-Max normalized range: [{dframe['Z_Normalized'].min():.3f}, {dframe['Z_Normalized'].max():.3f}]")
    
    # Shift Mean:
    dframe['NS_Normalized_Shifted'] = (dframe['NS_Normalized'] - np.mean(dframe['NS_Normalized']))
    dframe['EW_Normalized_Shifted'] = (dframe['EW_Normalized'] - np.mean(dframe['EW_Normalized']))
    dframe['Z_Normalized_Shifted'] = (dframe['Z_Normalized'] - np.mean(dframe['Z_Normalized']))
    print(f"New range for NS: [{dframe['NS_Normalized_Shifted'].min():.3f}, {dframe['NS_Normalized_Shifted'].max():.3f}]")
    print(f"New mean for NS: {dframe['NS_Normalized_Shifted'].mean():.6f}")
    print(f"New range for EW: [{dframe['EW_Normalized_Shifted'].min():.3f}, {dframe['EW_Normalized_Shifted'].max():.3f}]")
    print(f"New mean for EW: {dframe['EW_Normalized_Shifted'].mean():.6f}")
    print(f"New range for Z: [{dframe['Z_Normalized_Shifted'].min():.3f}, {dframe['Z_Normalized_Shifted'].max():.3f}]")
    print(f"New mean for Z: {dframe['Z_Normalized_Shifted'].mean():.6f}")
    
    # Plot:
    fig, axes = plt.subplots(3, 1, figsize=(10, 10))
    time_minutes = dframe['time (s)']/60
    # North-South data:
    axes[0].plot(time_minutes, dframe['NS_Normalized_Shifted'], label='North-South', color='forestgreen')
    axes[0].set_ylabel('Normalized Amplitude')
    axes[0].set_title('DNS North-South Data')
    axes[0].axhline(y=0, color='red', linestyle='--', label='y=0')
    axes[0].set_xlabel('Time (min)')
    axes[0].grid(True)
    
    # East-West data:
    axes[1].plot(time_minutes, dframe['EW_Normalized_Shifted'], label='East-West', color='rebeccapurple')
    axes[1].set_ylabel('Normalized Amplitude')
    axes[1].set_title('DNS East-West Data')
    axes[1].axhline(y=0, color='red', linestyle='--', label='y=0')
    axes[1].set_xlabel('Time (min)')
    axes[1].grid(True)
    
    # Z data:
    axes[2].plot(time_minutes, dframe['Z_Normalized_Shifted'], label='Z/Vertical Component', color='royalblue')
    axes[2].set_ylabel('Normalized Amplitude')
    axes[2].set_title('DNS Z-Component Data')
    axes[2].axhline(y=0, color='red', linestyle='--', label='y=0')
    axes[2].set_xlabel('Time (min)')
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()
    print("DNS = Detrended, Normalized, Shifted")
    print("")
    # Stats for plots
    print("North-South Stats:")
    print(ss.describe(dframe['NS_Normalized_Shifted']))
    print("median:", dframe["NS_Normalized_Shifted"].median())
    print("mode:", ss.mode(dframe["NS_Normalized_Shifted"])) 
    print("standard deviation:", dframe["NS_Normalized_Shifted"].std())
    print("")
    print("East-West Stats:")
    print(ss.describe(dframe['EW_Normalized_Shifted']))
    print("median:", dframe["EW_Normalized_Shifted"].median())
    print("mode:", ss.mode(dframe["EW_Normalized_Shifted"])) 
    print("standard deviation:", dframe["EW_Normalized_Shifted"].std())
    print("")
    print("Z-component Stats:")
    print(ss.describe(dframe['Z_Normalized_Shifted']))
    print("median:", dframe["Z_Normalized_Shifted"].median())
    print("mode:", ss.mode(dframe["Z_Normalized_Shifted"])) 
    print("standard deviation:", dframe["Z_Normalized_Shifted"].std())
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

***
### Tapering the Time Series
##### Notes:
- a Hann window/raised cosine was used to taper the DNS time series

In [ ]:
# Updated tapering function:
def apply_hann_window(data, NS_column, EW_column, Z_column, taper_fraction = 0.1):
    """Taper edges of time series with Hann window
    taper_fraction = 0.1 will start taper at 10% from edges"""
    n= len(data)
    taper_length = int(n * taper_fraction)

    # NS Data:
    # Create a copy of the data
    NS_windowed_data = data[NS_column].copy()
    NS_windowed_data = NS_windowed_data.astype(float) # needed for FFT code to work properly later on
    # Apply Hann window to the left edge
    NS_left_window = np.hanning(2 * taper_length)[:taper_length]
    NS_windowed_data.iloc[:taper_length] *= NS_left_window
    # Apply Hann window to the right edge
    NS_right_window = np.hanning(2 * taper_length)[taper_length:]
    NS_windowed_data.iloc[-taper_length:] *= NS_right_window

    # EW Data:
    # Create a copy of the data
    EW_windowed_data = data[EW_column].copy()
    EW_windowed_data = EW_windowed_data.astype(float) 
    # Apply Hann window to the left edge
    EW_left_window = np.hanning(2 * taper_length)[:taper_length]
    EW_windowed_data.iloc[:taper_length] *= EW_left_window
    # Apply Hann window to the right edge
    EW_right_window = np.hanning(2 * taper_length)[taper_length:]
    EW_windowed_data.iloc[-taper_length:] *= EW_right_window

    # Z Data:
    Z_windowed_data = data[Z_column].copy()
    Z_windowed_data = Z_windowed_data.astype(float) 
    # Apply Hann window to the left edge
    Z_left_window = np.hanning(2 * taper_length)[:taper_length]
    Z_windowed_data.iloc[:taper_length] *= Z_left_window
    # Apply Hann window to the right edge
    Z_right_window = np.hanning(2 * taper_length)[taper_length:]
    Z_windowed_data.iloc[-taper_length:] *= Z_right_window
    
    return NS_windowed_data, EW_windowed_data, Z_windowed_data

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# df:
NS_hann_result_df, EW_hann_result_df, Z_hann_result_df = apply_hann_window(df, 'NS_Normalized_Shifted', 'EW_Normalized_Shifted', 'Z_Normalized_Shifted')
# df2:
NS_hann_result_df2, EW_hann_result_df2, Z_hann_result_df2 = apply_hann_window(df2, 'NS_Normalized_Shifted', 'EW_Normalized_Shifted', 'Z_Normalized_Shifted')
# df3:
NS_hann_result_df3, EW_hann_result_df3, Z_hann_result_df3 = apply_hann_window(df3, 'NS_Normalized_Shifted', 'EW_Normalized_Shifted', 'Z_Normalized_Shifted')
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
NS_hann_results = [NS_hann_result_df, NS_hann_result_df2, NS_hann_result_df3]
EW_hann_results = [EW_hann_result_df, EW_hann_result_df2, EW_hann_result_df3]
Z_hann_results = [Z_hann_result_df, Z_hann_result_df2, Z_hann_result_df3]

# Plot tapered data with full data for all dataframes:
for i, dframe in enumerate(dataframes):
    if i == 0:
        print("dataframe: df")
    else:
        print(f"dataframe: df{i+1}")
    # Plot:
    fig, axes = plt.subplots(3, 1, figsize=(10, 10))
    time_minutes = dframe['time (s)']/60
    # North-South data:
    axes[0].plot(time_minutes, dframe['NS_Normalized_Shifted'], label='North-South (Full)', color='red')
    axes[0].plot(time_minutes, NS_hann_results[i], label='North-South (Tapered)', color='forestgreen') # make sure step for time_minutes=decimation_factor
    axes[0].set_ylabel('Normalized Amplitude')
    axes[0].set_title('Tapered and Full DNS North-South Data')
    axes[0].set_xlabel('Time (min)')
    axes[0].legend(loc='upper left')
    axes[0].grid(True)
    
    # East-West data:
    axes[1].plot(time_minutes, dframe['EW_Normalized_Shifted'], label='East-West (Full)', color='red')
    axes[1].plot(time_minutes, EW_hann_results[i], label='East-West (Tapered)', color='rebeccapurple')
    axes[1].set_ylabel('Normalized Amplitude')
    axes[1].set_title('Tapered and Full DNS East-West Data')
    axes[1].set_xlabel('Time (min)')
    axes[1].legend(loc='upper left')
    axes[1].grid(True)
    
    # Z data:
    axes[2].plot(time_minutes, dframe['Z_Normalized_Shifted'], label='Z (Full)', color='red')
    axes[2].plot(time_minutes, Z_hann_results[i], label='Z (Tapered)', color='royalblue')
    axes[2].set_ylabel('Normalized Amplitude')
    axes[2].set_title('Tapered and Full DNS Z-Component Data')
    axes[2].set_xlabel('Time (min)')
    axes[2].legend(loc='upper left')
    #axes[2].set_xlim(-0.001,0.01)
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Hann window one column at a time:
def apply_hann_window1(data, column, taper_fraction = 0.1):
    """Taper edges of time series with Hann window
    taper_fraction = 0.1 will start taper at 10% from edges"""
    n= len(data)
    taper_length = int(n * taper_fraction)
    
    # Create a copy of the data
    windowed_data = data[column].copy()
    windowed_data = windowed_data.astype(float) # needed for FFT code to work properly
    
    # Apply Hann window to the left edge
    left_window = np.hanning(2 * taper_length)[:taper_length]
    windowed_data[:taper_length] *= left_window
    
    # Apply Hann window to the right edge
    right_window = np.hanning(2 * taper_length)[taper_length:]
    windowed_data[-taper_length:] *= right_window
    
    return windowed_data

#print(type(apply_hann_window1(df, "NS_Normalized_Shifted")))

***
### Combining Subsampling and Tapering
##### Notes:
- The full series was tapered before subsampling
- A Hann window was used for tapering
- Use if needed

In [ ]:
def subsample_windowed(NS_windowed, EW_windowed, Z_windowed, sr=1024, decimation_factor=5):
    """Downsample Hann tapered data.
    NS_windowed: tapered North-South data, type: pandas series
    EW_windowed: tapered East-West data, pandas series
    Z_windowed: tapered Up-Down data, pandas series
    sr: sample rate in Hz, int
    decimation_factor: step size for downsampling, int"""
    NS_decimated_data = scipy.signal.decimate(NS_windowed, decimation_factor, ftype='iir', zero_phase=True)
    EW_decimated_data = scipy.signal.decimate(EW_windowed, decimation_factor, ftype='iir', zero_phase=True)
    Z_decimated_data = scipy.signal.decimate(Z_windowed, decimation_factor, ftype='iir', zero_phase=True)
    new_sr = sr/decimation_factor
    return NS_decimated_data, EW_decimated_data, Z_decimated_data, new_sr
    
# Results for all dataframes:  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# df:
NS_decimated_data_df, EW_decimated_data_df, Z_decimated_data_df, new_sr_df = subsample_windowed(NS_hann_result_df, EW_hann_result_df, Z_hann_result_df)
# df2:
NS_decimated_data_df2, EW_decimated_data_df2, Z_decimated_data_df2, new_sr_df2 = subsample_windowed(NS_hann_result_df2, EW_hann_result_df2, Z_hann_result_df2)
# df3:
NS_decimated_data_df3, EW_decimated_data_df3, Z_decimated_data_df3, new_sr_df3 = subsample_windowed(NS_hann_result_df3, EW_hann_result_df3, Z_hann_result_df3)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
NS_taper_sub = [NS_decimated_data_df, NS_decimated_data_df2, NS_decimated_data_df3]
EW_taper_sub = [EW_decimated_data_df, EW_decimated_data_df2, EW_decimated_data_df3]
Z_taper_sub = [Z_decimated_data_df, Z_decimated_data_df2, Z_decimated_data_df3]

# for i, dframe in enumerate(dataframes):
#     if i ==0:
#         print("dataframe: df")
#     else:
#         print(f"dataframe: df{i+1}")
#     # Plot:
#     fig, axes = plt.subplots(3, 1, figsize=(12, 12))
#     time_minutes = dframe['time (s)']/60
#     # North-South data:
#     axes[0].plot(time_minutes, dframe['NS_Normalized_Shifted'], label='North-South (Full)', color='red')
#     axes[0].plot(time_minutes[::5], NS_taper_sub[i], label='Tapered North-South (Step=5)', color='forestgreen') # make sure step for time_minutes=decimation_factor
#     axes[0].set_ylabel('North-South Component')
#     axes[0].set_title('Tapered/Subsampled and Full DNS North-South Data')
#     axes[0].set_xlabel('Time (min)')
#     axes[0].legend(loc='upper left')
#     axes[0].grid(True)
    
#     # East-West data:
#     axes[1].plot(time_minutes, dframe['EW_Normalized_Shifted'], label='East-West (Full)', color='red')
#     axes[1].plot(time_minutes[::5], EW_taper_sub[i], label='Tapered East-West (Step=5)', color='rebeccapurple')
#     axes[1].set_ylabel('East-West Component')
#     axes[1].set_title('Tapered/Subsampled and Full DNS East-West Data')
#     axes[1].set_xlabel('Time (min)')
#     axes[1].legend(loc='upper left')
#     axes[1].grid(True)
    
#     # Z data:
#     axes[2].plot(time_minutes, dframe['Z_Normalized_Shifted'], label='Z (Full)', color='red')
    # axes[2].plot(time_minutes[::5], Z_taper_sub[i], label='Tapered Z (Step=5)', color='royalblue')
    # axes[2].set_ylabel('Z Component')
    # axes[2].set_title('Tapered/Subsampled and Full DNS Z-Component Data')
    # axes[2].set_xlabel('Time (min)')
    # axes[2].legend(loc='upper left')
    # axes[2].grid(True)
    
    # plt.tight_layout()
    # plt.show()

# Don't need subsampling right now

***
### Short-Time Fourier Transform
##### Notes:
- Goal: apply moving window (20 second intervals)
- Using short time fourier transform from scipy.signal
- version 1: 20 second moving window, no window overlap
- version 2: 20 second moving window, 5 second step for overlaping windows
    - log frequency scale

In [ ]:
# version 1: no window overlap
def hann_windowed_fft_spectrogram(data, sr, plot_title, vmax, vmin, window_duration=20):
    """Compute FFT with scipy short time fourier transform
    data: Time series to perform STFT on (array)
    sr: sampling rate in Hz (int)
    plot_title: title for spectrogram plot (string)
    window_duration: window duration in seconds"""
    hop = int(sr*window_duration) # hop = number of samples in window_duration (20sec)
    # Generate hann window array with tapering only at edges
    window_array = np.ones(hop) 
    taper_length = int(hop * 0.1)  # 0.1 for tapering at 10% from edges
    
    # Apply taper to left edge
    left_window = np.hanning(2*taper_length)[:taper_length]
    window_array[:taper_length] = left_window
    
    # Apply taper to right edge
    right_window = np.hanning(2*taper_length)[taper_length:]
    window_array[-taper_length:] = right_window
    
    STF = ShortTimeFFT(win=window_array, hop=hop, fs=sr, fft_mode='onesided', mfft=None, scale_to='magnitude', phase_shift=0)
    STF_result = STF.stft(data)
    
    # Get time and frequency arrays
    time_axis = STF.t(len(data))  # Time values for each window (sec)
    time_minute = time_axis/60
    freq_axis = STF.f  # Frequency values
    
    # Plot spectrogram
    stft_plot = plt.figure(figsize=(12,4))
    plt.pcolormesh(time_minute, freq_axis, np.abs(STF_result), cmap='binary', shading='gouraud', vmax=vmax, vmin=vmin)
    plt.title(plot_title)
    plt.xlabel("Time (min)")
    plt.ylabel("Frequency (Hz)")
    cbar=plt.colorbar(label='Magnitude')
    cbar.formatter.set_powerlimits((0, 0))
    plt.grid(which='major')
    
    return stft_plot

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
for i, dframe in enumerate(dataframes):
    if i==0:
        print("dataframe: df")
    else:
        print(f"dataframe: df{i+1}")
    # Plot!------------------------------------------------------------
    # Spectrograms for NS data:
    print("Spectrogram for North-South data")
    # Full NS data:
    hann_windowed_fft_spectrogram(dframe['NS_Normalized_Shifted'].values, sr=original_sampling_rate, 
                             plot_title='Short Time Fourier Transform With 20s Hann Window (Full NS Data)', vmax=0.001, vmin=0, window_duration=20)
    if i==0:
        plt.ylim(top=150)
    else:
        plt.ylim(top=200)
    plt.tight_layout()
    plt.show()
    # Tapered/subsampled NS data: # Leaving out subsampled plots for now
    #hann_windowed_fft_spectrogram(NS_taper_sub[i], sr=new_sr_df, plot_title='Short Time Fourier Transform With 20s Hann Window (Tapered/Subsampled NS Data)',
     #                        vmax=0.001, vmin=0, window_duration=20) 
    # new sampling rate should be the same for all dataframes because they have been subsampled the same
    #plt.ylim(top=70)
    #plt.tight_layout()
    #plt.show()
    #---------------------------------------------------------------------------------
    # Spectrograms for EW data:
    print("Spectrogram for East-West data")
    # Full EW data:
    hann_windowed_fft_spectrogram(dframe['EW_Normalized_Shifted'].values, sr=original_sampling_rate, 
                             plot_title='Short Time Fourier Transform With 20s Hann Window (Full EW Data)', vmax=0.001, vmin=0, window_duration=20)
    if i==0:
        plt.ylim(top=150)
    else:
        plt.ylim(top=200)
    plt.tight_layout()
    plt.show()
    # Tapered/subsampled EW data:
    #hann_windowed_fft_spectrogram(EW_taper_sub[i], sr=new_sr_df, plot_title='Short Time Fourier Transform With 20s Hann Window (Tapered/Subsampled EW Data)',
     #                        vmax=0.001, vmin=0, window_duration=20)
    #plt.ylim(top=70)
    #plt.tight_layout()
    #plt.show()
    #--------------------------------------------------------------------------------
    # Spectrograms for Z-component data:
    print("Spectrogram for Z-component data")
    # Full Z data:
    hann_windowed_fft_spectrogram(dframe['Z_Normalized_Shifted'].values, sr=original_sampling_rate, 
                             plot_title='Short Time Fourier Transform With 20s Hann Window (Full Z Data)', vmax=0.001, vmin=0, window_duration=20)
    if i==0:
        plt.ylim(top=150)
    else:
        plt.ylim(top=200)
    plt.tight_layout()
    plt.show()
    # Tapered/subsampled Z data:
    #hann_windowed_fft_spectrogram(Z_taper_sub[i], sr=new_sr_df, plot_title='Short Time Fourier Transform With 20s Hann Window (Tapered/Subsampled Z Data)',
     #                        vmax=0.001, vmin=0, window_duration=20)
    #plt.ylim(top=70)
    #plt.tight_layout()
    #plt.show()
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

In [ ]:
# Spectrograms ver. 2: overlapping windows
# -still using 20 second moving window, but want 5 second step so adjacent windows overlap
def hann_windowed_fft_spectrogram2(data, sr, plot_title, vmax, vmin, window_duration=20, step_duration=5):
    """Compute FFT with scipy short time fourier transform and plot
    data: Time series to perform STFT on (array)
    sr: sampling rate in Hz (int)
    plot_title: title for spectrogram plot (string)
    window_duration: window duration in seconds
    step_duration: step size between windows in seconds"""
    hop = int(sr * window_duration)   # number of samples in the 20sec window
    step = int(sr * step_duration)    # number of samples to advance each step (5sec)

    # Generate hann window array with tapering only at edges
    window_array = np.ones(hop)  # Start with flat window
    taper_length = int(hop * 0.1)  # 0.1 for tapering at 10% from edges

    # Apply taper to left edge
    left_window = np.hanning(2*taper_length)[:taper_length]
    window_array[:taper_length] = left_window

    # Apply taper to right edge
    right_window = np.hanning(2*taper_length)[taper_length:]
    window_array[-taper_length:] = right_window

    STF = ShortTimeFFT(win=window_array, hop=step, fs=sr, fft_mode='onesided', mfft=None, scale_to='magnitude', phase_shift=0)
    STF_result = STF.stft(data) # perform the STFT

    # Get time and frequency arrays
    time_axis = STF.t(len(data))  # Time values for each window (sec)
    time_minute = time_axis / 60
    freq_axis = STF.f  # Frequency values

    # Plot spectrogram
    stft_plot = plt.figure(figsize=(12, 4))
    #plt.pcolormesh(time_minute, freq_axis, np.abs(STF_result), cmap='jet', shading='gouraud', vmax=vmax, vmin=vmin)
    # try to plot log version:
    plt.pcolormesh(time_minute, freq_axis, np.log10(np.abs(STF_result)), cmap='jet', shading='gouraud', vmax=vmax, vmin=vmin)
    plt.yscale('log')
    plt.title(plot_title)
    plt.xlabel("Time (min)")
    plt.ylabel("Frequency (Hz)")
    cbar = plt.colorbar(label='log\u2081\u2080(Magnitude)')
    #cbar.formatter.set_powerlimits((0, 0))
    plt.ylim(top=sr//2)
    plt.grid(which='major')
    plt.grid(which='minor')

    return stft_plot
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
for i, dframe in enumerate(dataframes):
    if i==0:
        print("dataframe: df")
    else:
        print(f"dataframe: df{i+1}")
    # Plot!------------------------------------------------------------
    # Spectrograms for NS data:
    print("Spectrogram for North-South data")
    # Full NS data:
    hann_windowed_fft_spectrogram2(dframe['NS_Normalized_Shifted'].values, sr=original_sampling_rate, 
                             plot_title='Short Time Fourier Transform With 20s Hann Window (Full NS Data)', vmax=-4, vmin=-8, window_duration=20)
    plt.tight_layout()
    plt.show()
    # Tapered/subsampled NS data: # leaving out subsampled plots for now
    #hann_windowed_fft_spectrogram2(NS_taper_sub[i], sr=new_sr_df, plot_title='Short Time Fourier Transform With 20s Hann Window (Tapered/Subsampled NS Data)',
     #                        vmax=0.001, vmin=0, window_duration=20) 
    # new sampling rate should be the same for all dataframes because they have been subsampled the same
    #plt.ylim(top=70)
    #plt.tight_layout()
    #plt.show()
    #---------------------------------------------------------------------------------
    # Spectrograms for EW data:
    print("Spectrogram for East-West data")
    # Full EW data:
    hann_windowed_fft_spectrogram2(dframe['EW_Normalized_Shifted'].values, sr=original_sampling_rate, 
                             plot_title='Short Time Fourier Transform With 20s Hann Window (Full EW Data)', vmax=-4, vmin=-8, window_duration=20)
    plt.tight_layout()
    plt.show()
    # Tapered/subsampled EW data:
   # hann_windowed_fft_spectrogram2(EW_taper_sub[i], sr=new_sr_df, plot_title='Short Time Fourier Transform With 20s Hann Window (Tapered/Subsampled EW Data)',
                        #     vmax=0.001, vmin=0, window_duration=20)
    #plt.ylim(top=70)
    #plt.tight_layout()
    #plt.show()
    #--------------------------------------------------------------------------------
    # Spectrograms for Z-component data
    print("Spectrogram for Z-component data")
    # Full Z data:
    hann_windowed_fft_spectrogram2(dframe['Z_Normalized_Shifted'].values, sr=original_sampling_rate, 
                             plot_title='Short Time Fourier Transform With 20s Hann Window (Full Z Data)', vmax=-4, vmin=-8, window_duration=20)
    plt.tight_layout()
    plt.show()
    # Tapered/subsampled Z data:
    #hann_windowed_fft_spectrogram2(Z_taper_sub[i], sr=new_sr_df, plot_title='Short Time Fourier Transform With 20s Hann Window (Tapered/Subsampled Z Data)',
                  #           vmax=0.001, vmin=0, window_duration=20)
    #plt.ylim(top=70)
    #plt.tight_layout()
    #plt.show()
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

***
#### Continuous Wavelet Transform
- Plot CWT scalograms to compare to STFT spectrogram results
- The Morlet wavelet is used for all CWT plots

In [ ]:
original_sampling_rate = 1024
def cwt_scalogram(dframe, NS_col, EW_col, Z_col, time_col, vmin, vmax, sr=original_sampling_rate):
    """Plot CWT scalograms with Morlet wavelet using NS, EW, and Z component data.
    dframe: dataframe of interest
    NS_col: North-South data column
    EW_col: East-West data column
    Z_col: Z-component data column
    time_col: time data column
    sr: sampling rate"""
    # Setup Data:
    NS_signal = dframe[NS_col].values
    EW_signal = dframe[EW_col].values
    Z_signal = dframe[Z_col].values
    time = dframe[time_col].values
    time_minutes = time / 60
    
    downsample_factor = 5
    NS_signal_ds = signal.decimate(NS_signal, downsample_factor)
    EW_signal_ds = signal.decimate(EW_signal, downsample_factor)
    Z_signal_ds = signal.decimate(Z_signal, downsample_factor)
    sampling_rate_ds = sr // downsample_factor
    time_ds = time_minutes[::downsample_factor]
    
    # Setup Resolution:
    nv = 32 # voices per octave == nv
    
    # Perform CWT: 
    NS_Wx, NS_frequencies = sq.cwt(NS_signal_ds, 'morlet', nv=nv, fs=sampling_rate_ds) # North-South
    EW_Wx, EW_frequencies = sq.cwt(EW_signal_ds, 'morlet', nv=nv, fs=sampling_rate_ds) # East-West
    Z_Wx, Z_frequencies = sq.cwt(Z_signal_ds, 'morlet', nv=nv, fs=sampling_rate_ds) # Up-Down (Z)
    print(f"Sample rate: {sampling_rate_ds} Hz")
    print(f"Nyquist: {sampling_rate_ds / 2} Hz")
    
    # Max frequency should never exceed Nyquist
    # Limit to relevant frequency range:
    NS_freq_mask = (NS_frequencies >=1) & (NS_frequencies <= sampling_rate_ds/2)
    NS_freqs_plot = NS_frequencies[NS_freq_mask]
    NS_Wx_masked = NS_Wx[NS_freq_mask, :]  
    EW_freq_mask = (EW_frequencies >= 1) & (EW_frequencies <= sampling_rate_ds/2)
    EW_freqs_plot = EW_frequencies[EW_freq_mask]
    EW_Wx_masked = EW_Wx[EW_freq_mask, :]
    Z_freq_mask = (Z_frequencies >=1) & (Z_frequencies <= sampling_rate_ds/2)
    Z_freqs_plot = Z_frequencies[Z_freq_mask]
    Z_Wx_masked = Z_Wx[Z_freq_mask, :]
    
    # Downsample in time for display?
    time_skip = 50
    NS_Wx_display = np.abs(NS_Wx_masked[:, ::time_skip]) # takes the entire frequency axis, but jumps across the time axis in steps = time_skip
    EW_Wx_display = np.abs(EW_Wx_masked[:, ::time_skip])
    Z_Wx_display = np.abs(Z_Wx_masked[:, ::time_skip])
    time_display = time_ds[::time_skip] # need same jump for time labels so graph still matches data correctly
    
    # Calculate log magnitude
    NS_Wx_log = np.log10(NS_Wx_display)
    EW_Wx_log = np.log10(EW_Wx_display)
    Z_Wx_log = np.log10(Z_Wx_display)

    # Option: Define the "floor" for the data
    # e.g. anything quieter than threshold_val will be treated as "Not a Number" (NaN), which renders as transparent/white.
    threshold_val = vmin
    NS_Wx_log[NS_Wx_log < threshold_val] = np.nan
    EW_Wx_log[EW_Wx_log < threshold_val] = np.nan
    Z_Wx_log[Z_Wx_log < threshold_val] = np.nan

    # Plot scalogram for each direction:
    fig1 = plt.figure(figsize=(14, 6))
    #plt.pcolormesh(time_display, NS_frequencies, NS_Wx_log, cmap='jet', shading='auto')#, vmax=-4.5, vmin=-6.0)
    plt.pcolormesh(time_display, NS_freqs_plot, NS_Wx_log, cmap='jet', shading='auto', vmin=vmin, vmax=vmax)
    plt.yscale('log')
    plt.ylabel('Frequency (Hz)')
    plt.xlabel('Time (min)')
    plt.title('North-South CWT')
    plt.colorbar(label='log\u2081\u2080(Magnitude)')
    plt.show()

    fig2 = plt.figure(figsize=(14, 6))
    #plt.pcolormesh(time_display, EW_frequencies, EW_Wx_log, cmap='jet', shading='auto')#, vmax=-4.5, vmin=-6.0)
    plt.pcolormesh(time_display, EW_freqs_plot, EW_Wx_log, cmap='jet', shading='auto', vmin=vmin, vmax=vmax)
    plt.yscale('log')
    plt.ylabel('Frequency (Hz)')
    plt.xlabel('Time (min)')
    plt.title('East-West CWT')
    plt.colorbar(label='log\u2081\u2080(Magnitude)')
    plt.show()

    fig3 = plt.figure(figsize=(14, 6))
    #plt.pcolormesh(time_display, Z_frequencies, Z_Wx_log, cmap='jet', shading='auto')#, vmax=-4.5, vmin=-6.0)
    plt.pcolormesh(time_display, Z_freqs_plot, Z_Wx_log, cmap='jet', shading='auto', vmin=vmin, vmax=vmax)
    plt.yscale('log')
    plt.ylabel('Frequency (Hz)')
    plt.xlabel('Time (min)')
    plt.title('Up-Down CWT')
    plt.colorbar(label='log\u2081\u2080(Magnitude)')
    plt.show()

    return fig1, fig2, fig3
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("DataFrame: df")
cwt_scalogram(df, 'NS_Normalized_Shifted', 'EW_Normalized_Shifted', 'Z_Normalized_Shifted', 'time (s)', vmin=-5, vmax=-2, sr=original_sampling_rate)
print("~"*100)
print("DataFrame: df2")
cwt_scalogram(df2, 'NS_Normalized_Shifted', 'EW_Normalized_Shifted', 'Z_Normalized_Shifted', 'time (s)', vmin=-4.5, vmax=-1.5, sr=original_sampling_rate)
print("~"*100)
print("DataFrame: df3")
cwt_scalogram(df3, 'NS_Normalized_Shifted', 'EW_Normalized_Shifted', 'Z_Normalized_Shifted', 'time (s)', vmin=-4, vmax=-1, sr=original_sampling_rate)